# Adani Power stock price forecasting

This notebook loads the included historical closing-price data, fits a Facebook Prophet time-series model, and forecasts the next 30 days.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from prophet import Prophet

DATA_PATH = Path("Adani_Power_Historical_Data.csv")

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"])
required_columns = {"Date", "Close"}
missing_columns = required_columns.difference(prices.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

prices = prices[["Date", "Close"]].dropna().sort_values("Date")
if prices.empty or (prices["Close"] <= 0).any():
    raise ValueError("Close prices must contain at least one positive value.")

prophet_data = prices.rename(columns={"Date": "ds", "Close": "y"})
prophet_data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(prophet_data["ds"], prophet_data["y"], label="Closing price")
ax.set(title="Adani Power historical closing price", xlabel="Date", ylabel="Price")
ax.legend()
plt.show()

In [ ]:
model = Prophet()
model.fit(prophet_data)
future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(30)

In [ ]:
model.plot(forecast)
plt.title("Adani Power closing-price forecast")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

model.plot_components(forecast)
plt.show()